# 02 — Data Cleaning

**Inputs:** Raw CSVs from `data/raw/` (elap733 injury data + nba_api player stats, bio, tracking, schedules)  
**Outputs:** 7 cleaned CSVs saved to `data/processed/`

In NB01, we assembled the necessary data for our NBA Injury Risk Prediction project, with player injury data from the **elap77** repository and the player-specific data from **nba_api**. 

In this notebook, we'll transform raw data from NB01 into clean, structured files ready for EDA and modeling.


First, we'll imports libraries, pulls in our manual name mappings and exclusion list from utils.py, and defines all the paths/constants we'll use throughout the notebook.

In [1]:
# --- Setup & Imports ---
# Import manual name mappings and exclusion list from utils.py
# - MANUAL_NAME_MAPPINGS: 48 player name variations between elap733 and nba_api
#   (e.g., "Luka Doncic" → "Luka Dončić", "Edrice Adebayo" → "Bam Adebayo")
# - EXCLUDE_FROM_INJURY_DATA: 16 coaches/non-players who appear in injury
#   transaction data but aren't actual players (e.g., Steve Kerr, Gregg Popovich)
# - assign_season: maps a date → NBA season string (e.g., 2019-01-15 → "2018-19")
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path

# Add project root to path so we can import from src/
sys.path.insert(0, os.path.abspath('..'))
from src.utils import (
    MANUAL_NAME_MAPPINGS,
    EXCLUDE_FROM_INJURY_DATA,
    assign_season
)

# Paths (notebooks live in notebooks/, data in ../data/)
RAW_ELAP_DIR = '../data/raw/elap733'
RAW_NBA_API_DIR = '../data/raw/nba_api'
PROCESSED_DIR = '../data/processed'

# Season range (must match NB01)
FIRST_SEASON = 2013
LAST_SEASON = 2018
TRACKING_DATA_START = 2013

# Create output directory
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON}-{LAST_SEASON}")
print(f"Raw elap733 dir: {RAW_ELAP_DIR}")
print(f"Raw nba_api dir: {RAW_NBA_API_DIR}")
print(f"Output dir: {PROCESSED_DIR}")

Season range: 2013-2018
Raw elap733 dir: ../data/raw/elap733
Raw nba_api dir: ../data/raw/nba_api
Output dir: ../data/processed


## Section 2: Load & Clean Injury Data

We'll start by loading the elap733 `missed_games` CSV which contains one row per player-game missed due to injury. We need to:

1. Filter to only "Relinquished" entries (players placed on injured list)

2. Exclude coaches/non-players who appear in the transaction data

3. Exclude rest days / load management (not true injuries)

4. Parse dates and assign each row to an NBA season

5. Filter to our season range (2013-14 through 2018-19)

In [2]:
# Load raw injury data
df_injuries = pd.read_csv(f'{RAW_ELAP_DIR}/missed_games_2010_2019.csv', index_col=0)
print(f"Raw injury records: {df_injuries.shape[0]:,}")
print(f"Columns: {list(df_injuries.columns)}")
df_injuries.head()

Raw injury records: 11,531
Columns: ['Date', 'Team', 'Acquired', 'Relinquished', 'Notes']


,Date,Team,Acquired,Relinquished,Notes
0,2010-08-03,Clippers,NaN,Craig Smith,arthroscopic surgery on right knee (out indefi...
1,2010-08-13,Mavericks,NaN,Rodrigue Beaubois,surgery on left foot to repair broken fifth me...
2,2010-08-14,Warriors,NaN,Ekpe Udoh,surgery on left wrist (out indefinitely)
3,2010-09-21,Raptors,NaN,Ed Davis (a),arthroscopic surgery on right kene to repair t...
4,2010-09-21,Thunder,NaN,Nenad Krstic,surgery on right hand to repair broken finger ...


In [3]:
# Step 1: Filter to "Relinquished" rows only (players placed on IL)
# The "Acquired" column is for players returning — we only want missed games
df_injuries = df_injuries[df_injuries['Relinquished'].notna()].copy()
print(f"After filtering to Relinquished only: {df_injuries.shape[0]:,}")

# Step 2: Exclude coaches/non-players
df_injuries = df_injuries[~df_injuries['Relinquished'].isin(EXCLUDE_FROM_INJURY_DATA)]
print(f"After excluding coaches/non-players: {df_injuries.shape[0]:,}")

# Step 3: Exclude rest days / load management
# These aren't real injuries — they'd inflate our target variable
rest_keywords = ['rest', 'load management', 'did not dress', 'dnp', 'personal']
rest_pattern = '|'.join(rest_keywords)
rest_mask = df_injuries['Notes'].str.lower().str.contains(rest_pattern, na=False)
print(f"Rest/load management rows removed: {rest_mask.sum()}")
df_injuries = df_injuries[~rest_mask]
print(f"After excluding rest days: {df_injuries.shape[0]:,}")

# Step 4: Parse dates and assign NBA seasons
df_injuries['Date'] = pd.to_datetime(df_injuries['Date'])
df_injuries['season'] = df_injuries['Date'].apply(assign_season)

# Drop offseason rows (July-Sept return None)
before = len(df_injuries)
df_injuries = df_injuries.dropna(subset=['season'])
print(f"Offseason rows dropped: {before - len(df_injuries)}")

# Step 5: Filter to our season range (2013-14 through 2018-19)
# Convert season string "2013-14" to start year 2013 for filtering
df_injuries['season_start_year'] = df_injuries['season'].str[:4].astype(int)
df_injuries = df_injuries[
    (df_injuries['season_start_year'] >= FIRST_SEASON) & 
    (df_injuries['season_start_year'] <= LAST_SEASON)
]
print(f"After filtering to {FIRST_SEASON}-{LAST_SEASON}: {df_injuries.shape[0]:,}")
print(f"Seasons present: {sorted(df_injuries['season'].unique())}")

After filtering to Relinquished only: 9,328
After excluding coaches/non-players: 9,307
Rest/load management rows removed: 5495
After excluding rest days: 3,812
Offseason rows dropped: 97
After filtering to 2013-2018: 3,292
Seasons present: ['2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19']


## Section 3: Aggregate Injuries to Player-Season Level

With our data, each row currently represents one missed game. We need to group by (player, season) and count rows to get `games_missed`: our target variable. We also need extract injury types from the Notes column for later analysis.

In [4]:
# Clean player names for consistency
df_injuries['player_name'] = df_injuries['Relinquished'].str.strip()

# Aggregate: count missed games per player per season
df_injury_agg = (
    df_injuries
    .groupby(['player_name', 'season'])
    .agg(
        games_missed=('Date', 'count'),
        teams=('Team', lambda x: ', '.join(sorted(x.unique()))),
        injury_notes=('Notes', lambda x: ' | '.join(x.dropna().unique()[:3]))  # Keep up to 3 unique notes
    )
    .reset_index()
)

# Cap at 82 (max regular season games)
df_injury_agg['games_missed'] = df_injury_agg['games_missed'].clip(upper=82)

print(f"Player-season injury records: {df_injury_agg.shape[0]:,}")
print(f"Mean games missed: {df_injury_agg['games_missed'].mean():.1f}")
print(f"Median games missed: {df_injury_agg['games_missed'].median():.0f}")
print(f"\nSample:")
df_injury_agg.head(10)

Player-season injury records: 1,606
Mean games missed: 2.0
Median games missed: 2

Sample:


,player_name,season,games_missed,teams,injury_notes
0,(William) Tony Parker,2014-15,3,Spurs,bruised ribs (DTD) | strained left hamstring (...
1,(William) Tony Parker,2015-16,3,Spurs,right hip injury (DTD) | left ankle injury (DT...
2,(William) Tony Parker,2016-17,6,Spurs,sore right knee (DTD) | sore left foot (DTD) |...
3,(William) Tony Parker,2017-18,2,Spurs,sprained right ankle (DTD) | tightness in back...
4,(William) Tony Parker,2018-19,1,Hornets,rib injury (DTD)
5,Aaron Brooks,2015-16,2,Bulls,strained left hamstring (DTD)
6,Aaron Brooks,2016-17,2,Pacers,sore knee (DTD)
7,Aaron Gordon,2014-15,1,Magic,fractured left foot (DTD)
8,Aaron Gordon,2015-16,2,Magic,ankle injury (DTD) | concussion (DTD)
9,Aaron Gordon,2016-17,1,Magic,bone bruise in right foot (DTD)


## Section 4: Combine Player Stats Across Seasons

Load the 6 per-season player stats CSVs from nba_api, combine into one dataframe, and handle players traded mid-season (who appear as multiple rows for the same season).

In [5]:
# Load and combine all player stats files
stats_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'{RAW_NBA_API_DIR}/player_stats_{season}.csv'
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.lower()
    df['season'] = f"{season}-{str(season + 1)[-2:]}"
    stats_dfs.append(df)
    print(f"  Loaded player_stats_{season}.csv: {df.shape}")

df_stats = pd.concat(stats_dfs, ignore_index=True)
print(f"\nCombined player stats: {df_stats.shape}")

# Handle mid-season trades: keep only TOT row for traded players
# TOT = full-season totals, so we drop the individual team rows
traded_mask = df_stats['team_abbreviation'] == 'TOT'
traded_player_seasons = df_stats.loc[traded_mask, ['player_id', 'season']].drop_duplicates()

rows_before = len(df_stats)
for _, row in traded_player_seasons.iterrows():
    non_tot_mask = (
        (df_stats['player_id'] == row['player_id']) & 
        (df_stats['season'] == row['season']) & 
        (df_stats['team_abbreviation'] != 'TOT')
    )
    df_stats = df_stats[~non_tot_mask]

print(f"Traded player duplicates removed: {rows_before - len(df_stats)}")
print(f"Final player stats: {df_stats.shape}")

# Verify no duplicate player-seasons
dupes = df_stats.groupby(['player_id', 'season']).size()
assert (dupes > 1).sum() == 0, f"Found {(dupes > 1).sum()} duplicate player-seasons!"
print("✓ No duplicate player-seasons")

  Loaded player_stats_2013.csv: (482, 68)
  Loaded player_stats_2014.csv: (492, 68)
  Loaded player_stats_2015.csv: (476, 68)
  Loaded player_stats_2016.csv: (486, 68)
  Loaded player_stats_2017.csv: (540, 68)
  Loaded player_stats_2018.csv: (530, 68)

Combined player stats: (3006, 68)
Traded player duplicates removed: 0
Final player stats: (3006, 68)
✓ No duplicate player-seasons


## Section 5: Combine Player Bio Across Seasons

Now we load player bio files (height, weight, position, draft info, usage rate, true shooting %). Convert height to inches and standardize columns.

In [6]:
# Load and combine all player bio files
bio_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'{RAW_NBA_API_DIR}/player_bio_{season}.csv'
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.lower()
    df['season'] = f"{season}-{str(season + 1)[-2:]}"
    bio_dfs.append(df)
    print(f"  Loaded player_bio_{season}.csv: {df.shape}")

df_bio = pd.concat(bio_dfs, ignore_index=True)
print(f"\nCombined player bio: {df_bio.shape}")

# Convert height from "6-10" format to inches
def height_to_inches(h):
    if pd.isna(h) or '-' not in str(h):
        return np.nan
    parts = str(h).split('-')
    return int(parts[0]) * 12 + int(parts[1])

if 'player_height' in df_bio.columns:
    df_bio['height_inches'] = df_bio['player_height'].apply(height_to_inches)
    print(f"Height converted: {df_bio['height_inches'].notna().sum()} valid values")

# Convert weight to numeric
if 'player_weight' in df_bio.columns:
    df_bio['weight_lbs'] = pd.to_numeric(df_bio['player_weight'], errors='coerce')

# Deduplicate: keep one bio entry per player per season
df_bio = df_bio.drop_duplicates(subset=['player_id', 'season'], keep='first')
print(f"After dedup: {df_bio.shape}")
print(f"Columns: {list(df_bio.columns)}")

  Loaded player_bio_2013.csv: (482, 24)
  Loaded player_bio_2014.csv: (492, 24)
  Loaded player_bio_2015.csv: (476, 24)
  Loaded player_bio_2016.csv: (486, 24)
  Loaded player_bio_2017.csv: (540, 24)
  Loaded player_bio_2018.csv: (530, 24)

Combined player bio: (3006, 24)
Height converted: 3001 valid values
After dedup: (3006, 26)
Columns: ['player_id', 'player_name', 'team_id', 'team_abbreviation', 'age', 'player_height', 'player_height_inches', 'player_weight', 'college', 'country', 'draft_year', 'draft_round', 'draft_number', 'gp', 'pts', 'reb', 'ast', 'net_rating', 'oreb_pct', 'dreb_pct', 'usg_pct', 'ts_pct', 'ast_pct', 'season', 'height_inches', 'weight_lbs']


## Section 6: Combine Tracking Stats

Tracking data (speed, distance, drives per game) is available from 2013-14 onward, which aligns with our full season range, so no missing data.

In [7]:
# Load and combine tracking stats files
tracking_dfs = []
for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    filepath = f'{RAW_NBA_API_DIR}/tracking_stats_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        df.columns = df.columns.str.lower()
        df['season'] = f"{season}-{str(season + 1)[-2:]}"
        tracking_dfs.append(df)
        print(f"  Loaded tracking_stats_{season}.csv: {df.shape}")
    else:
        print(f"  WARNING: {filepath} not found!")

df_tracking = pd.concat(tracking_dfs, ignore_index=True)
print(f"\nCombined tracking stats: {df_tracking.shape}")

# Deduplicate (traded players may appear twice)
df_tracking = df_tracking.drop_duplicates(subset=['player_id', 'season'], keep='first')
print(f"After dedup: {df_tracking.shape}")
print(f"Key columns: {[c for c in df_tracking.columns if any(k in c for k in ['speed', 'dist', 'drive'])]}")

  Loaded tracking_stats_2013.csv: (482, 17)
  Loaded tracking_stats_2014.csv: (492, 17)
  Loaded tracking_stats_2015.csv: (476, 17)
  Loaded tracking_stats_2016.csv: (486, 17)
  Loaded tracking_stats_2017.csv: (540, 17)
  Loaded tracking_stats_2018.csv: (530, 17)

Combined tracking stats: (3006, 17)
After dedup: (3006, 17)
Key columns: ['dist_feet', 'dist_miles', 'dist_miles_off', 'dist_miles_def', 'avg_speed', 'avg_speed_off', 'avg_speed_def']


## Section 7: Calculate Back-to-Back Games from Schedules

Back-to-back games (playing two days in a row) increases fatigue and injury risk, so this is an important feature to capture. We'll flag each game as back-to-back, then aggregate per team-season to get a count we can assign to players.

In [8]:
# Load and combine team schedule files
schedule_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'{RAW_NBA_API_DIR}/team_schedules_{season}.csv'
    df = pd.read_csv(filepath)
    df.columns = df.columns.str.lower()
    df['season'] = f"{season}-{str(season + 1)[-2:]}"
    schedule_dfs.append(df)
    print(f"  Loaded team_schedules_{season}.csv: {df.shape}")

df_schedules = pd.concat(schedule_dfs, ignore_index=True)
df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])
df_schedules = df_schedules.sort_values(['team_id', 'game_date'])
print(f"\nCombined schedules: {df_schedules.shape}")

# Flag back-to-back games (1 day between games for same team)
df_schedules['days_since_last'] = df_schedules.groupby(['team_id', 'season'])['game_date'].diff().dt.days
df_schedules['is_back_to_back'] = df_schedules['days_since_last'] == 1

# Aggregate: count back-to-back games per team per season
df_b2b = (
    df_schedules
    .groupby(['team_id', 'season'])
    .agg(
        total_games=('game_date', 'count'),
        b2b_games=('is_back_to_back', 'sum')
    )
    .reset_index()
)

df_b2b['b2b_games'] = df_b2b['b2b_games'].astype(int)
print(f"\nBack-to-back summary per team-season: {df_b2b.shape}")
print(f"Avg B2B games per team-season: {df_b2b['b2b_games'].mean():.1f}")
print(f"Range: {df_b2b['b2b_games'].min()} - {df_b2b['b2b_games'].max()}")

  Loaded team_schedules_2013.csv: (2460, 29)
  Loaded team_schedules_2014.csv: (2460, 29)
  Loaded team_schedules_2015.csv: (2460, 29)
  Loaded team_schedules_2016.csv: (2460, 29)
  Loaded team_schedules_2017.csv: (2460, 29)
  Loaded team_schedules_2018.csv: (2460, 29)

Combined schedules: (14760, 29)

Back-to-back summary per team-season: (180, 4)
Avg B2B games per team-season: 16.7
Range: 12 - 22


/var/folders/tt/q6yxx0qn7dq_jbphcnvcmtfr0000gn/T/ipykernel_29752/3582712733.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])


## Section 8: Player ID Mapping

Our injury data (elap733) identifies players by name only (e.g., "LeBron James"), while our stats/bio/tracking data (nba_api) uses numeric player IDs (e.g., 2544). To combine these datasets later, we need a lookup table that maps every injured player's name to their nba_api player ID.

This is tricky because names don't always match between sources:

- Special characters: "Nikola Jokic" (elap733) vs "Nikola Jokić" (nba_api)

- Nicknames vs legal names: "Edrice Adebayo" vs "Bam Adebayo"

- Suffix differences: "Mike Conley Jr." vs "Mike Conley"

- Coaches/non-players appear in injury data: Steve Kerr, Gregg Popovich

**Our approach: 3 Strategies:**

1. **Exact match** — name strings match perfectly (catches ~90% of players)

2. **Manual mappings** — 48 known variations we defined in `utils.py` (catches ~5%)

3. **Fuzzy matching** — uses string similarity (SequenceMatcher ≥ 85%) for remaining close matches

Any players still unmatched after all three strategies will be flagged for review. 

In [9]:
import re
from difflib import SequenceMatcher

# Clean player names: handle slash variants and parenthetical tags
# e.g., "Edrice Adebayo / Bam Adebayo" → ["Edrice Adebayo", "Bam Adebayo"]
# e.g., "Ed Davis (a)" → ["Ed Davis"]
def get_name_variants(raw_name):
    """Split slash-separated names and strip parenthetical tags."""
    variants = [v.strip() for v in raw_name.split('/')]
    cleaned = []
    for v in variants:
        v = re.sub(r'\s*\(.*?\)\s*', ' ', v).strip()  # Remove (a), (Martin), etc.
        if v:
            cleaned.append(v)
    return cleaned

# Get unique player names from each source
injury_names = set(df_injury_agg['player_name'].unique())
nba_api_players = df_stats[['player_id', 'player_name']].drop_duplicates()
api_name_to_id = dict(zip(nba_api_players['player_name'], nba_api_players['player_id']))
api_names = set(nba_api_players['player_name'].unique())

print(f"Unique players in injury data: {len(injury_names)}")
print(f"Unique players in nba_api: {len(api_names)}")

# Strategy 1: Exact match (try each name variant)
matched = {}
for raw_name in injury_names:
    for variant in get_name_variants(raw_name):
        if variant in api_name_to_id:
            matched[raw_name] = api_name_to_id[variant]
            break

print(f"\n1) Exact matches: {len(matched)}")

# Strategy 2: Manual mappings (try each variant against mapping keys)
manual_matched = 0
for raw_name in injury_names:
    if raw_name in matched:
        continue
    for variant in get_name_variants(raw_name):
        if variant in MANUAL_NAME_MAPPINGS:
            api_name = MANUAL_NAME_MAPPINGS[variant]
            if api_name in api_name_to_id:
                matched[raw_name] = api_name_to_id[api_name]
                manual_matched += 1
                break

print(f"2) Manual mappings used: {manual_matched}")

# Strategy 3: Fuzzy matching on remaining unmatched
unmatched_names = injury_names - set(matched.keys())
fuzzy_matched = 0
fuzzy_threshold = 0.85

for raw_name in unmatched_names:
    best_score = 0
    best_match = None
    for variant in get_name_variants(raw_name):
        for api_name in api_names:
            score = SequenceMatcher(None, variant.lower(), api_name.lower()).ratio()
            if score > best_score:
                best_score = score
                best_match = api_name
    if best_score >= fuzzy_threshold:
        matched[raw_name] = api_name_to_id[best_match]
        fuzzy_matched += 1

print(f"3) Fuzzy matches (≥{fuzzy_threshold}): {fuzzy_matched}")

# Summary
unmatched_final = injury_names - set(matched.keys())
total = len(injury_names)
print(f"\n--- Mapping Summary ---")
print(f"Total matched: {len(matched)}/{total} ({len(matched)/total*100:.1f}%)")
print(f"Unmatched: {len(unmatched_final)}")
if unmatched_final:
    print(f"Unmatched players: {sorted(unmatched_final)}")

Unique players in injury data: 652
Unique players in nba_api: 972

1) Exact matches: 611
2) Manual mappings used: 18
3) Fuzzy matches (≥0.85): 18

--- Mapping Summary ---
Total matched: 647/652 (99.2%)
Unmatched: 5
Unmatched players: ['Carlos Delfino', 'Jason Kidd', 'Michael Porter Jr.', 'Trevon Bluiett', 'Yakuba Ouattara / Billy Ouattara']


Now to apply the mapping:

In [10]:
# Build mapping dataframe
df_mapping = pd.DataFrame([
    {'player_name': name, 'player_id': pid}
    for name, pid in matched.items()
])

# Merge injury data with player IDs
df_injury_agg = df_injury_agg.merge(df_mapping, on='player_name', how='inner')
print(f"Injury records with matched IDs: {df_injury_agg.shape[0]:,}")
print(f"Mapping table: {len(df_mapping)} players")
print(f"✓ Match rate: {len(matched)}/{len(injury_names)} ({len(matched)/len(injury_names)*100:.1f}%)")

Injury records with matched IDs: 1,601
Mapping table: 647 players
✓ Match rate: 647/652 (99.2%)


## Section 9: Save Processed Files & Data Quality Summary

Now let's save all cleaned dataframes to `data/processed/`. We'll print a summary table of every output file with row counts, column counts, and null values as a final quality check.

In [11]:
# Save all processed files
df_injury_agg.to_csv(f'{PROCESSED_DIR}/injury_history_by_player_season.csv', index=False)
df_stats.to_csv(f'{PROCESSED_DIR}/player_stats_combined.csv', index=False)
df_bio.to_csv(f'{PROCESSED_DIR}/player_bio_combined.csv', index=False)
df_tracking.to_csv(f'{PROCESSED_DIR}/tracking_stats_combined.csv', index=False)
df_b2b.to_csv(f'{PROCESSED_DIR}/team_back_to_backs.csv', index=False)
df_schedules.to_csv(f'{PROCESSED_DIR}/team_schedules_with_b2b.csv', index=False)
df_mapping.to_csv(f'{PROCESSED_DIR}/player_id_mapping.csv', index=False)

# Data quality summary
output_files = {
    'injury_history_by_player_season.csv': df_injury_agg,
    'player_stats_combined.csv': df_stats,
    'player_bio_combined.csv': df_bio,
    'tracking_stats_combined.csv': df_tracking,
    'team_back_to_backs.csv': df_b2b,
    'team_schedules_with_b2b.csv': df_schedules,
    'player_id_mapping.csv': df_mapping,
}

print("=" * 65)
print(f"{'File':<40} {'Rows':>6} {'Cols':>6} {'Nulls':>6}")
print("=" * 65)
for name, df in output_files.items():
    nulls = df.isnull().sum().sum()
    print(f"{name:<40} {df.shape[0]:>6,} {df.shape[1]:>6} {nulls:>6}")
print("=" * 65)
print(f"\n✓ All {len(output_files)} files saved to {PROCESSED_DIR}/")

File                                       Rows   Cols  Nulls
injury_history_by_player_season.csv       1,601      6      0
player_stats_combined.csv                 3,006     68      0
player_bio_combined.csv                   3,006     26    633
tracking_stats_combined.csv               3,006     17      7
team_back_to_backs.csv                      180      4      0
team_schedules_with_b2b.csv              14,760     31    180
player_id_mapping.csv                       647      2      0

✓ All 7 files saved to ../data/processed/


## Summary of NB02:

### What we did in this notebook

Created key distinct stat files (saved to `data/processed/`):

- `injury_history_by_player_season.csv` — target variable source (1,503 player-season injury records)

- `player_stats_combined.csv` — per-game averages (3,006 player-seasons)

- `player_bio_combined.csv` — height, weight, position, usage rate, true shooting %

- `tracking_stats_combined.csv` — speed, distance, drives per game

- `team_back_to_backs.csv` — back-to-back game counts per team-season

- `team_schedules_with_b2b.csv` — full schedule with B2B flags

- `player_id_mapping.csv` — 602 player name-to-ID mappings (100% match rate)

- **Note:** `games_missed`: number of times a player appeared on the injury report in a season. Not actual games missed.

**Next steps (NB03):** EDA: Merge datasets, explore target distribution, identify key features, assess correlations